# Transformer Locator

This notebook builds a smooth synthetic magnetic field on the unit sphere, trains a transformer conditional density model for the endpoint of a noisy magnetic trajectory, and estimates a mutual-information lower bound

$$I(\text{endpoint}; \text{trajectory}) \gtrsim h(\text{endpoint}) - h(\text{endpoint}\mid\text{trajectory}).$$

The endpoint prior is uniform on the unit sphere, so `h(endpoint) = log2(4*pi)` bits with respect to unit-sphere area measure. If the same densities are written over Earth surface area, both entropy terms receive the same `2*log2(R)` shift, so the mutual information is unchanged. The Earth radius `R` is used below to convert geodesic step lengths in meters to angular steps.

## 1. Preparations

In [1]:
import math
import random
from dataclasses import dataclass

import numpy as np
import plotly.graph_objects as go
import torch
from IPython.display import clear_output, display
from torch import nn
from torch.nn import functional as F

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float64

# Geometry controls.
EARTH_RADIUS_M = 6_371_000.0
STEP_METERS = 1.0
STEP_RAD = STEP_METERS / EARTH_RADIUS_M

# Noise controls for observation tokens. A trajectory with N moves is represented
# as B0, du0, B1, du1, ..., B_N; the final token has no outgoing step, so its du slot is 0.
# The walk uses angular steps internally, but observed 2D tangent steps are
# represented in meters so 1 m steps remain numerically visible to the model.
EPS_B_STD = 0.003
EPS_U_STD = 0.05

# Random-walk controls.
MIN_STEPS = 1
MAX_STEPS = 16
WALK_DIRECTION_RANDOMNESS = 0.45

# Model/training controls. Increase these for a better estimate after checking runtime.
N_COMPONENTS = 10
D_MODEL = 192
N_HEADS = 8
N_LAYERS = 6
DROPOUT = 0.04
TRAIN_EPOCHS = 800
TRAIN_BATCH_SIZE = 512
LEARNING_RATE = 1.2e-4
AUX_DIRECTION_WEIGHT = 0.20
PLOT_EVERY = 10

# Kent normalizer controls. Larger quadrature gives a better likelihood but costs more.
KENT_QUAD_POINTS = 384
KAPPA_MIN = 1e-4
KAPPA_MAX = 1e14
KENT_ASYMPTOTIC_KAPPA = 120.0
BETA_FRACTION = 0.92

EVAL_STEP_COUNTS = [1, 2, 4, 8, 12, 16]
EVAL_BATCH_SIZE = 512
EVAL_BATCHES_PER_POINT = 8

print(f"device={DEVICE}, step={STEP_METERS:.1f} m, angular step={STEP_RAD:.3e} rad")

device=cuda, step=1.0 m, angular step=1.570e-07 rad


In [2]:
def normalize(x, eps=1e-8):
    return x / x.norm(dim=-1, keepdim=True).clamp_min(eps)


def sample_uniform_sphere(n, device=DEVICE):
    return normalize(torch.randn(n, 3, device=device, dtype=DTYPE))


def tangent_basis(u):
    flat = u.reshape(-1, 3)
    z = torch.tensor([0.0, 0.0, 1.0], device=flat.device, dtype=flat.dtype).expand_as(
        flat
    )
    x = torch.tensor([1.0, 0.0, 0.0], device=flat.device, dtype=flat.dtype).expand_as(
        flat
    )
    ref = torch.where(flat[:, 2:3].abs() < 0.9, z, x)
    e1 = normalize(torch.cross(ref, flat, dim=-1))
    e2 = torch.cross(flat, e1, dim=-1)
    return e1.reshape_as(u), e2.reshape_as(u)


def exp_map_sphere(u, tangent_step):
    theta = tangent_step.norm(dim=-1, keepdim=True)
    direction = tangent_step / theta.clamp_min(1e-8)
    moved = torch.cos(theta) * u + torch.sin(theta) * direction
    return normalize(moved)


FIELD_CENTERS = normalize(
    torch.tensor(
        [
            [0.22, -0.61, 0.76],
            [-0.78, 0.34, 0.52],
            [0.57, 0.73, -0.38],
            [-0.18, -0.83, -0.53],
        ],
        dtype=DTYPE,
    )
)
FIELD_MOMENTS = normalize(
    torch.tensor(
        [
            [0.88, 0.17, -0.44],
            [-0.31, 0.92, 0.24],
            [0.12, -0.58, 0.81],
            [-0.70, -0.21, -0.68],
        ],
        dtype=DTYPE,
    )
)
FIELD_STRENGTHS = torch.tensor([1.1, -0.8, 0.65, -0.55], dtype=DTYPE)
WAVE_DIRS = normalize(
    torch.tensor(
        [
            [1.0, 0.3, -0.2],
            [-0.4, 1.0, 0.5],
            [0.2, -0.7, 1.0],
            [0.8, -0.1, 0.6],
            [-0.5, -0.6, 0.7],
            [0.3, 0.9, 0.4],
        ],
        dtype=DTYPE,
    )
)
FIELD_FEATURE_SCALES_M = torch.tensor(
    [50_000.0, 10_000.0, 2_000.0, 400.0, 80.0, 20.0, 5.0, 1.25], dtype=DTYPE
)
FIELD_FEATURE_WEIGHTS = torch.tensor(
    [0.12, 0.10, 0.08, 0.06, 0.045, 0.035, 0.028, 0.022], dtype=DTYPE
)


def B(u):
    """Synthetic magnetic field B(u) -> (Bx, By, Bz) for unit 3D vector u."""
    shape = u.shape
    flat = normalize(u.reshape(-1, 3))
    centers = FIELD_CENTERS.to(flat.device)
    moments = FIELD_MOMENTS.to(flat.device)
    strengths = FIELD_STRENGTHS.to(flat.device)
    dirs = WAVE_DIRS.to(flat.device)
    scales = FIELD_FEATURE_SCALES_M.to(flat.device)
    weights = FIELD_FEATURE_WEIGHTS.to(flat.device)

    r = flat[:, None, :] - 0.55 * centers[None, :, :]
    r2 = (r * r).sum(dim=-1).clamp_min(1e-4)
    mr = (moments[None, :, :] * r).sum(dim=-1)
    dipoles = strengths[None, :, None] * (
        3.0 * r * mr[:, :, None] / r2[:, :, None] ** 2.5
        - moments[None, :, :] / r2[:, :, None] ** 1.5
    )
    dipole_field = dipoles.sum(dim=1)

    projected_m = EARTH_RADIUS_M * (flat @ dirs.T)
    fine = torch.zeros_like(flat)
    for j, (scale, weight) in enumerate(zip(scales, weights, strict=True)):
        p0 = 2.0 * math.pi * projected_m[:, j % dirs.shape[0]] / scale
        p1 = 2.0 * math.pi * projected_m[:, (j + 2) % dirs.shape[0]] / scale
        p2 = 2.0 * math.pi * projected_m[:, (j + 4) % dirs.shape[0]] / scale
        fine[:, 0] = fine[:, 0] + weight * torch.sin(p0 + 0.37 * j)
        fine[:, 1] = fine[:, 1] + weight * torch.cos(p1 + 0.53 * j)
        fine[:, 2] = fine[:, 2] + weight * torch.sin(p2 - 0.29 * j)

    # The coordinate-like term gives a globally identifiable backbone, while the
    # multi-scale waves add smooth meter-scale structure for 1 m trajectories.
    field = 0.70 * flat + 0.12 * dipole_field + fine
    return field.reshape(shape)


def fibonacci_sphere(n, device=DEVICE):
    i = torch.arange(n, device=device, dtype=DTYPE) + 0.5
    z = 1.0 - 2.0 * i / n
    phi = math.pi * (3.0 - math.sqrt(5.0)) * i
    r = torch.sqrt((1.0 - z * z).clamp_min(0.0))
    return torch.stack([r * torch.cos(phi), r * torch.sin(phi), z], dim=-1)


def make_trajectories(batch_size, step_counts=None, device=DEVICE, return_path=False):
    if step_counts is None:
        counts = torch.randint(MIN_STEPS, MAX_STEPS + 1, (batch_size,), device=device)
    elif isinstance(step_counts, int):
        counts = torch.full((batch_size,), step_counts, device=device, dtype=torch.long)
    else:
        counts = torch.as_tensor(step_counts, device=device, dtype=torch.long)
        batch_size = int(counts.numel())

    max_steps = int(counts.max().item())
    max_tokens = max_steps + 1
    x = torch.zeros(batch_size, max_tokens, 5, device=device, dtype=DTYPE)
    pad_mask = torch.ones(batch_size, max_tokens, device=device, dtype=torch.bool)

    u = sample_uniform_sphere(batch_size, device=device)
    path = torch.zeros(batch_size, max_tokens, 3, device=device, dtype=DTYPE)
    path[:, 0, :] = u
    e1, e2 = tangent_basis(u)
    angle = 2.0 * math.pi * torch.rand(batch_size, 1, device=device, dtype=DTYPE)
    heading = torch.cos(angle) * e1 + torch.sin(angle) * e2

    for t in range(max_steps):
        token_active = t <= counts
        step_active = t < counts
        e1, e2 = tangent_basis(u)
        heading = normalize(heading - (heading * u).sum(dim=-1, keepdim=True) * u)
        heading_2d = torch.stack(
            [(heading * e1).sum(dim=-1), (heading * e2).sum(dim=-1)], dim=-1
        )
        noisy_direction = heading_2d + WALK_DIRECTION_RANDOMNESS * torch.randn(
            batch_size, 2, device=device, dtype=DTYPE
        )
        du_2d_meters = STEP_METERS * normalize(noisy_direction)
        du_2d_radians = du_2d_meters / EARTH_RADIUS_M
        tangent_step = du_2d_radians[:, 0:1] * e1 + du_2d_radians[:, 1:2] * e2

        x[token_active, t, :3] = (
            B(u) + EPS_B_STD * torch.randn(batch_size, 3, device=device, dtype=DTYPE)
        )[token_active]
        x[step_active, t, 3:] = (
            du_2d_meters
            + EPS_U_STD * torch.randn(batch_size, 2, device=device, dtype=DTYPE)
        )[step_active]
        pad_mask[token_active, t] = False

        next_u = exp_map_sphere(u, tangent_step)
        transported_heading = normalize(
            tangent_step - (tangent_step * next_u).sum(dim=-1, keepdim=True) * next_u
        )
        u = torch.where(step_active[:, None], next_u, u)
        heading = torch.where(step_active[:, None], transported_heading, heading)
        path[:, t + 1, :] = u

    final_active = max_steps <= counts
    x[final_active, max_steps, :3] = (
        B(u) + EPS_B_STD * torch.randn(batch_size, 3, device=device, dtype=DTYPE)
    )[final_active]
    pad_mask[final_active, max_steps] = False

    if return_path:
        return x, pad_mask, u, counts, path
    return x, pad_mask, u, counts


with torch.no_grad():
    demo_u = sample_uniform_sphere(5)
    print(B(demo_u).detach().cpu())

tensor([[-0.0638,  0.7860,  0.1525],
        [-0.8850, -0.3402, -0.3774],
        [-0.4284, -0.4094,  1.1205],
        [ 0.0643,  0.2341, -0.6931],
        [-0.0499,  0.5011, -0.1979]], dtype=torch.float64)


In [3]:
# Continuous field visualization on the sphere. RGB channels are Bx, By, Bz.
n_lat, n_lon = 72, 144
lat = np.linspace(-0.5 * math.pi, 0.5 * math.pi, n_lat)
lon = np.linspace(-math.pi, math.pi, n_lon, endpoint=False)
lat_grid, lon_grid = np.meshgrid(lat, lon, indexing="ij")
x_s = np.cos(lat_grid) * np.cos(lon_grid)
y_s = np.cos(lat_grid) * np.sin(lon_grid)
z_s = np.sin(lat_grid)
sphere_pts = np.stack([x_s, y_s, z_s], axis=-1).reshape(-1, 3)

faces_i, faces_j, faces_k = [], [], []
for a in range(n_lat - 1):
    for b in range(n_lon):
        p00 = a * n_lon + b
        p01 = a * n_lon + (b + 1) % n_lon
        p10 = (a + 1) * n_lon + b
        p11 = (a + 1) * n_lon + (b + 1) % n_lon
        faces_i.extend([p00, p01])
        faces_j.extend([p10, p10])
        faces_k.extend([p01, p11])

with torch.no_grad():
    field = B(torch.as_tensor(sphere_pts, device=DEVICE, dtype=DTYPE)).cpu().numpy()

lo = np.quantile(field, 0.02, axis=0)
hi = np.quantile(field, 0.98, axis=0)
field_rgb = np.clip((field - lo) / (hi - lo + 1e-12), 0.0, 1.0)
basis = np.array([[255, 0, 0], [0, 255, 0], [0, 0, 255]], dtype=float)
rgb = np.clip(field_rgb @ basis, 0, 255).astype(int)
vertexcolor = [f"rgb({r},{g},{b})" for r, g, b in rgb]

fig = go.Figure(
    data=[
        go.Mesh3d(
            x=sphere_pts[:, 0],
            y=sphere_pts[:, 1],
            z=sphere_pts[:, 2],
            i=faces_i,
            j=faces_j,
            k=faces_k,
            vertexcolor=vertexcolor,
            flatshading=False,
            lighting=dict(ambient=0.72, diffuse=0.35, specular=0.08),
            name="B component mixture",
            showscale=False,
        )
    ]
)
fig.update_layout(
    title="Synthetic magnetic field components on the unit sphere (red=Bx, green=By, blue=Bz)",
    scene=dict(aspectmode="data"),
    margin=dict(l=0, r=0, t=45, b=0),
)
fig.show()

In [4]:
# Visualize one 16-step biased random walk sample on the sphere.
with torch.no_grad():
    _, _, _, _, sample_path = make_trajectories(1, step_counts=16, return_path=True)
path_pts = sample_path[0].cpu().numpy()
center = path_pts.mean(axis=0)
center = center / np.linalg.norm(center)
north = np.array([0.0, 0.0, 1.0])
ref = north if abs(center[2]) < 0.9 else np.array([1.0, 0.0, 0.0])
east = np.cross(ref, center)
east = east / np.linalg.norm(east)
north_local = np.cross(center, east)
local_xy_m = EARTH_RADIUS_M * np.column_stack([path_pts @ east, path_pts @ north_local])
span_m = max(np.ptp(local_xy_m[:, 0]), np.ptp(local_xy_m[:, 1]), 8.0 * STEP_METERS)
patch_radius_m = 3.0 * span_m
patch_radius_rad = patch_radius_m / EARTH_RADIUS_M
patch_mask = np.arccos(np.clip(sphere_pts @ center, -1.0, 1.0)) <= patch_radius_rad
patch_idx = np.flatnonzero(patch_mask)
patch_lookup = -np.ones(len(sphere_pts), dtype=int)
patch_lookup[patch_idx] = np.arange(len(patch_idx))
patch_face_mask = (
    patch_mask[np.asarray(faces_i)]
    & patch_mask[np.asarray(faces_j)]
    & patch_mask[np.asarray(faces_k)]
)
patch_i = patch_lookup[np.asarray(faces_i)[patch_face_mask]]
patch_j = patch_lookup[np.asarray(faces_j)[patch_face_mask]]
patch_k = patch_lookup[np.asarray(faces_k)[patch_face_mask]]
patch_pts = sphere_pts[patch_idx]
patch_colors = [vertexcolor[idx] for idx in patch_idx]
camera_eye = 1.000004 * center

fig = go.Figure()
fig.add_trace(
    go.Mesh3d(
        x=patch_pts[:, 0],
        y=patch_pts[:, 1],
        z=patch_pts[:, 2],
        i=patch_i,
        j=patch_j,
        k=patch_k,
        vertexcolor=patch_colors,
        opacity=0.40,
        flatshading=False,
        lighting=dict(ambient=0.82, diffuse=0.25, specular=0.03),
        name="B component mixture",
        showscale=False,
    )
)
fig.add_trace(
    go.Scatter3d(
        x=path_pts[:, 0],
        y=path_pts[:, 1],
        z=path_pts[:, 2],
        mode="lines+markers",
        line=dict(color="black", width=7),
        marker=dict(size=4, color=np.arange(path_pts.shape[0]), colorscale="Inferno"),
        name="16-step walk",
    )
)
fig.add_trace(
    go.Scatter3d(
        x=[path_pts[0, 0], path_pts[-1, 0]],
        y=[path_pts[0, 1], path_pts[-1, 1]],
        z=[path_pts[0, 2], path_pts[-1, 2]],
        mode="markers",
        marker=dict(size=[7, 8], color=["lime", "red"]),
        name="start/end",
    )
)
fig.update_layout(
    title=f"Random biased walk sample: 16 steps x {STEP_METERS:.1f} m, zoomed patch",
    scene=dict(
        aspectmode="data",
        camera=dict(eye=dict(x=camera_eye[0], y=camera_eye[1], z=camera_eye[2])),
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
    ),
    margin=dict(l=0, r=0, t=45, b=0),
)
fig.show()

## 2. Model

The transformer consumes padded sequences encoding `B0, du0, B1, du1, ..., B_N`. In tensor form each token is `(Bx, By, Bz, du_1, du_2)`; the final endpoint token has `du=(0, 0)` because there is no outgoing step. The model emits a mixture of Kent distributions over the unit-sphere endpoint. The conditional entropy estimate is the train NLL averaged over fresh synthetic trajectories generated each epoch and reported in bits.

In [5]:
class SinusoidalPositionEncoding(nn.Module):
    def __init__(self, d_model, max_len=256):
        super().__init__()
        position = torch.arange(max_len, dtype=DTYPE).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=DTYPE) * (-math.log(10_000.0) / d_model)
        )
        pe = torch.zeros(max_len, d_model, dtype=DTYPE)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.shape[1], :]


class KentMixtureTransformer(nn.Module):
    def __init__(
        self,
        n_components=N_COMPONENTS,
        d_model=D_MODEL,
        n_heads=N_HEADS,
        n_layers=N_LAYERS,
    ):
        super().__init__()
        self.n_components = n_components
        self.register_buffer(
            "input_scale",
            torch.tensor([1.0, 1.0, 1.0, STEP_METERS, STEP_METERS], dtype=DTYPE),
        )
        self.cls = nn.Parameter(torch.zeros(1, 1, d_model, dtype=DTYPE))
        self.length_embedding = nn.Embedding(MAX_STEPS + 2, d_model, dtype=DTYPE)
        self.input = nn.Sequential(
            nn.Linear(5, 2 * d_model),
            nn.LayerNorm(2 * d_model),
            nn.GELU(),
            nn.Linear(2 * d_model, d_model),
            nn.LayerNorm(d_model),
        )
        self.pos = SinusoidalPositionEncoding(d_model, max_len=MAX_STEPS + 2)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=DROPOUT,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(4 * d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, n_components + n_components * 6),
        )

    def forward(self, x, pad_mask):
        h = self.input(x / self.input_scale)
        cls = self.cls.expand(x.shape[0], -1, -1)
        h = torch.cat([cls, h], dim=1)
        h = self.pos(h)
        cls_mask = torch.zeros(x.shape[0], 1, device=x.device, dtype=torch.bool)
        full_pad_mask = torch.cat([cls_mask, pad_mask], dim=1)
        h = self.encoder(h, src_key_padding_mask=full_pad_mask)
        lengths = (~pad_mask).sum(dim=1).clamp(max=MAX_STEPS)
        pooled = h[:, 0, :] + self.length_embedding(lengths)
        raw = self.head(pooled)
        logits = raw[:, : self.n_components]
        comp = raw[:, self.n_components :].reshape(-1, self.n_components, 6)
        gamma1 = normalize(comp[..., :3])
        log_kappa = torch.clamp(
            comp[..., 3], min=math.log(KAPPA_MIN), max=math.log(KAPPA_MAX)
        )
        kappa = torch.exp(log_kappa)
        beta = 0.5 * BETA_FRACTION * kappa * torch.tanh(comp[..., 4])
        angle = math.pi * torch.tanh(comp[..., 5])
        return logits, gamma1, kappa, beta, angle


def kent_axes(gamma1, angle):
    e1, e2 = tangent_basis(gamma1)
    c = torch.cos(angle).unsqueeze(-1)
    s = torch.sin(angle).unsqueeze(-1)
    gamma2 = c * e1 + s * e2
    gamma3 = -s * e1 + c * e2
    return gamma2, gamma3


KENT_QUAD_Z_NP, KENT_QUAD_W_NP = np.polynomial.legendre.leggauss(KENT_QUAD_POINTS)
KENT_QUAD_Z = torch.as_tensor(KENT_QUAD_Z_NP, device=DEVICE, dtype=DTYPE)
KENT_QUAD_W = torch.as_tensor(KENT_QUAD_W_NP, device=DEVICE, dtype=DTYPE)


def kent_log_normalizer(gamma1, kappa, beta, angle):
    z = KENT_QUAD_Z[None, None, :]
    w = KENT_QUAD_W[None, None, :]
    one_minus_z2 = (1.0 - z.square()).clamp_min(0.0)
    bessel_arg = beta[:, :, None] * one_minus_z2
    log_i0 = (
        torch.log(torch.special.i0e(bessel_arg).clamp_min(torch.finfo(DTYPE).tiny))
        + bessel_arg.abs()
    )
    log_terms = kappa[:, :, None] * z + log_i0 + torch.log(w)
    log_quad = math.log(2.0 * math.pi) + torch.logsumexp(log_terms, dim=-1)

    # For very sharp Kent densities, Gauss-Legendre nodes do not resolve the tiny
    # cap around the mode. The local tangent-plane Gaussian normalizer is stable.
    log_asymptotic = (
        kappa
        + math.log(2.0 * math.pi)
        - 0.5
        * (
            torch.log((kappa - 2.0 * beta).clamp_min(KAPPA_MIN))
            + torch.log((kappa + 2.0 * beta).clamp_min(KAPPA_MIN))
        )
    )
    return torch.where(kappa > KENT_ASYMPTOTIC_KAPPA, log_asymptotic, log_quad)


def kent_mixture_log_prob(endpoint, logits, gamma1, kappa, beta, angle):
    gamma2, gamma3 = kent_axes(gamma1, angle)
    y = endpoint[:, None, :]
    d1 = (y * gamma1).sum(dim=-1)
    d2 = (y * gamma2).sum(dim=-1)
    d3 = (y * gamma3).sum(dim=-1)
    log_unnorm = kappa * d1 + beta * (d2.square() - d3.square())
    log_component = log_unnorm - kent_log_normalizer(gamma1, kappa, beta, angle)
    return torch.logsumexp(F.log_softmax(logits, dim=-1) + log_component, dim=-1)


def nll_loss(model, x, pad_mask, endpoint):
    params = model(x, pad_mask)
    return -kent_mixture_log_prob(endpoint, *params).mean()


def kent_mixture_mean_direction(logits, gamma1):
    weights = F.softmax(logits, dim=-1)
    return normalize((weights[..., None] * gamma1).sum(dim=1))


def training_objective(model, x, pad_mask, endpoint):
    params = model(x, pad_mask)
    logits, gamma1, _, _, _ = params
    nll = -kent_mixture_log_prob(endpoint, *params).mean()
    pred_dir = kent_mixture_mean_direction(logits, gamma1)
    aux_direction = (1.0 - (pred_dir * endpoint).sum(dim=-1)).mean()
    return nll + AUX_DIRECTION_WEIGHT * aux_direction, nll, aux_direction


model = KentMixtureTransformer().to(device=DEVICE, dtype=DTYPE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=TRAIN_EPOCHS, eta_min=0.1 * LEARNING_RATE
)
sum(p.numel() for p in model.parameters())

C:\Users\shich\AppData\Local\Temp\ipykernel_13936\2331374294.py:50: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)


3543238

In [6]:
loss_history = []
objective_history = []
aux_history = []
model.train()

for epoch in range(1, TRAIN_EPOCHS + 1):
    x, pad_mask, endpoint, counts = make_trajectories(TRAIN_BATCH_SIZE)
    loss, nll, aux_direction = training_objective(model, x, pad_mask, endpoint)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()

    objective_history.append(float(loss.detach().cpu()))
    loss_history.append(float(nll.detach().cpu()))
    aux_history.append(float(aux_direction.detach().cpu()))
    if epoch == 1 or epoch % PLOT_EVERY == 0 or epoch == TRAIN_EPOCHS:
        clear_output(wait=True)
        loss_bits = [value / math.log(2.0) for value in loss_history]
        fig = go.Figure()
        fig.add_trace(
            go.Scatter(
                x=list(range(1, len(loss_bits) + 1)),
                y=loss_bits,
                mode="lines",
                name="train NLL",
            )
        )
        fig.add_trace(
            go.Scatter(
                x=list(range(1, len(aux_history) + 1)),
                y=aux_history,
                mode="lines",
                name="direction aux",
                yaxis="y2",
            )
        )
        fig.update_layout(
            title=f"Training loss, epoch {epoch}/{TRAIN_EPOCHS}",
            xaxis_title="epoch",
            yaxis_title="NLL = estimated h(endpoint | trajectory), bits",
            yaxis2=dict(title="direction aux", overlaying="y", side="right"),
            height=360,
            margin=dict(l=40, r=20, t=50, b=40),
        )
        display(fig)

print(f"final train NLL: {loss_history[-1] / math.log(2.0):.4f} bits")

final train NLL: -1.1609 bits


## 3. Evaluation

For each fixed trajectory length, the model NLL estimates `h(endpoint | trajectory)`. The endpoint prior is uniform, so the lower-bound estimate is `log2(4*pi) - NLL/log(2)` in bits.

In [7]:
@torch.no_grad()
def estimate_conditional_entropy(step_count):
    model.eval()
    weighted_loss = 0.0
    total = 0
    for _ in range(EVAL_BATCHES_PER_POINT):
        x, pad_mask, endpoint, _ = make_trajectories(
            EVAL_BATCH_SIZE, step_counts=step_count
        )
        loss = nll_loss(model, x, pad_mask, endpoint)
        weighted_loss += float(loss.cpu()) * EVAL_BATCH_SIZE
        total += EVAL_BATCH_SIZE
    return weighted_loss / total


h_endpoint_unit_bits = math.log2(4.0 * math.pi)
h_endpoint_earth_surface_bits = math.log2(4.0 * math.pi * EARTH_RADIUS_M**2)
print(f"h(endpoint) on unit sphere: {h_endpoint_unit_bits:.4f} bits")
print(
    f"h(endpoint) on Earth surface: {h_endpoint_earth_surface_bits:.4f} bits; MI is unchanged by this area-scale shift"
)

rows = []
for steps in EVAL_STEP_COUNTS:
    h_cond_bits = estimate_conditional_entropy(steps) / math.log(2.0)
    info_bits = h_endpoint_unit_bits - h_cond_bits
    rows.append(
        dict(
            steps=steps,
            distance_m=steps * STEP_METERS,
            h_cond_bits=h_cond_bits,
            info_bits=info_bits,
        )
    )

for row in rows:
    print(
        f"steps={row['steps']:2d}, distance={row['distance_m']:6.1f} m, "
        f"h_cond={row['h_cond_bits']:.4f} bits, I_lb={row['info_bits']:.4f} bits"
    )

h(endpoint) on unit sphere: 3.6515 bits
h(endpoint) on Earth surface: 48.8577 bits; MI is unchanged by this area-scale shift
steps= 1, distance=   1.0 m, h_cond=-1.1582 bits, I_lb=4.8097 bits
steps= 2, distance=   2.0 m, h_cond=-1.1480 bits, I_lb=4.7995 bits
steps= 4, distance=   4.0 m, h_cond=-1.1328 bits, I_lb=4.7843 bits
steps= 8, distance=   8.0 m, h_cond=-1.1150 bits, I_lb=4.7665 bits
steps=12, distance=  12.0 m, h_cond=-1.1579 bits, I_lb=4.8093 bits
steps=16, distance=  16.0 m, h_cond=-1.1995 bits, I_lb=4.8510 bits


In [8]:
steps = [r["steps"] for r in rows]
info_bits = [r["info_bits"] for r in rows]

fig = go.Figure()
fig.add_trace(go.Scatter(x=steps, y=info_bits, mode="lines+markers", name="bits"))
fig.update_layout(
    title="Estimated I(endpoint; trajectory) lower bound vs trajectory length",
    xaxis=dict(title="steps"),
    yaxis=dict(title="lower bound, bits"),
    height=420,
    margin=dict(l=50, r=55, t=50, b=45),
)
fig.show()